# 06. External validation — Part 4: frozen inference on NHANES + domain shift

## Что делаем в этой части

Part 3 показал, что NHANES можно сопоставить только с частью нашего frozen feature space:

- 19 / 37 признаков имеют прямой mapping;
- schema coverage ≈ 51%;
- у реального участника медианная заполненность ещё ниже.

Поэтому здесь мы **не называем результаты полноценной external accuracy validation для всех дефицитов**.

Вместо этого мы проводим честный **external transfer / stress test**:

1. загружаем frozen L1 + CatBoost + calibrators;
2. читаем frozen ensemble weights / thresholds из `04`;
3. запускаем модели на NHANES без дообучения;
4. считаем prediction prevalence и confidence;
5. оцениваем missingness shift;
6. оцениваем distribution/domain shift по напрямую сопоставленным признакам;
7. отдельно показываем результаты для участников, попадающих в возрастной диапазон внутренних данных;
8. маркируем низкое feature coverage как ограничение внешнего inference.

> Никакой tuning на NHANES в этой части не выполняется.


In [1]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from scipy.stats import ks_2samp

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

print("Python:", sys.version.split()[0])
print("Текущая рабочая папка:", Path.cwd())


Python: 3.11.16
Текущая рабочая папка: /home/sinopah/meditron/notebooks


## 1. Пути и frozen artifacts


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "deficiency_anemia.csv").exists():
            return candidate

    raise FileNotFoundError(
        "Не найден корень проекта с data/raw/deficiency_anemia.csv"
    )


PROJECT_ROOT = find_project_root()

INTERNAL_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "deficiency_anemia.csv"

BUNDLE_DIR = PROJECT_ROOT / "artifacts" / "inference_bundle"
MODEL_DIR = BUNDLE_DIR / "models"

EXTERNAL_MATRIX_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "nhanes_2021_2023"
    / "processed"
    / "nhanes_external_feature_matrix.csv"
)

EXTERNAL_ARTIFACTS_DIR = (
    PROJECT_ROOT / "artifacts" / "external_validation"
)

EXTERNAL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    INTERNAL_DATA_PATH,
    BUNDLE_DIR / "feature_contract.json",
    BUNDLE_DIR / "ensemble_meta_settings.csv",
    EXTERNAL_MATRIX_PATH,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Не найдены обязательные файлы:\n" + "\n".join(missing)
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("External matrix:", EXTERNAL_MATRIX_PATH)
print("Frozen bundle:", BUNDLE_DIR)


PROJECT_ROOT: /home/sinopah/meditron
External matrix: /home/sinopah/meditron/data/external/nhanes_2021_2023/processed/nhanes_external_feature_matrix.csv
Frozen bundle: /home/sinopah/meditron/artifacts/inference_bundle


## 2. Загружаем internal + NHANES + feature contract


In [3]:
internal_df = pd.read_csv(INTERNAL_DATA_PATH)
external_df = pd.read_csv(EXTERNAL_MATRIX_PATH)

with open(BUNDLE_DIR / "feature_contract.json", "r", encoding="utf-8") as f:
    feature_contract = json.load(f)

FEATURE_COLUMNS = feature_contract["features"]
TARGETS = feature_contract["targets"]
CATEGORICAL_FEATURES = feature_contract["categorical_features"]

assert len(FEATURE_COLUMNS) == 37

print("Internal:", internal_df.shape)
print("External:", external_df.shape)
print("Targets:", TARGETS)
print("Frozen features:", len(FEATURE_COLUMNS))


Internal: (840, 48)
External: (11933, 39)
Targets: ['iron_deficiency', 'B12_deficiency', 'folate_deficiency', 'B6_deficiency', 'copper_deficiency', 'inflammation_anemia']
Frozen features: 37


## 3. Проверяем frozen ensemble metadata

Сначала просто смотрим структуру файла из `04`.

Следующая ячейка умеет распознать несколько типичных вариантов названий колонок.
Если структура неожиданная — notebook остановится и покажет, что именно нужно поправить,
вместо того чтобы молча использовать неправильные веса.


In [4]:
ensemble_meta = pd.read_csv(
    BUNDLE_DIR / "ensemble_meta_settings.csv"
)

print("ensemble_meta_settings shape:", ensemble_meta.shape)
print("columns:", list(ensemble_meta.columns))

display(ensemble_meta)


ensemble_meta_settings shape: (30, 8)
columns: ['target', 'fold', 'w_l1', 'w_cb', 'w_rule', 'brier', 'threshold', 'train_f1_at_threshold']


,target,fold,w_l1,w_cb,w_rule,brier,threshold,train_f1_at_threshold
0,iron_deficiency,1,0.4,0.6,0,0.008309,0.41,0.989154
1,iron_deficiency,2,0.5,0.5,0,0.005232,0.40,0.993492
2,iron_deficiency,3,0.4,0.6,0,0.008164,0.42,0.989154
3,iron_deficiency,4,0.3,0.7,0,0.005798,0.48,0.993492
4,iron_deficiency,5,0.4,0.6,0,0.005268,0.33,0.993521
5,B12_deficiency,1,0.5,0.5,0,0.009698,0.66,0.976744
6,B12_deficiency,2,0.4,0.6,0,0.010754,0.26,0.973856
7,B12_deficiency,3,0.5,0.5,0,0.009794,0.30,0.980263
8,B12_deficiency,4,0.5,0.5,0,0.012189,0.26,0.970684
9,B12_deficiency,5,0.6,0.4,0,0.007857,0.23,0.983607


In [7]:
display(
    ensemble_meta[
        ensemble_meta["target"] == "iron_deficiency"
    ]
)

,target,fold,w_l1,w_cb,w_rule,brier,threshold,train_f1_at_threshold
0,iron_deficiency,1,0.4,0.6,0,0.008309,0.41,0.989154
1,iron_deficiency,2,0.5,0.5,0,0.005232,0.40,0.993492
2,iron_deficiency,3,0.4,0.6,0,0.008164,0.42,0.989154
3,iron_deficiency,4,0.3,0.7,0,0.005798,0.48,0.993492
4,iron_deficiency,5,0.4,0.6,0,0.005268,0.33,0.993521


In [8]:
def first_existing_column(columns, candidates):
    lookup = {str(c).lower(): c for c in columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


def extract_ensemble_settings(meta: pd.DataFrame, targets):
    cols = list(meta.columns)

    target_col = first_existing_column(
        cols,
        ["target", "task", "label"]
    )

    threshold_col = first_existing_column(
        cols,
        [
            "threshold",
            "ensemble_threshold",
            "best_threshold",
            "selected_threshold",
            "decision_threshold",
        ],
    )

    l1_weight_col = first_existing_column(
        cols,
        [
            "l1_weight",
            "weight_l1",
            "w_l1",
            "logreg_weight",
            "logistic_weight",
        ],
    )

    cat_weight_col = first_existing_column(
        cols,
        [
            "catboost_weight",
            "weight_catboost",
            "w_catboost",
            "w_cb",
            "cat_weight",
            "weight_cat",
        ],
    )

    rule_weight_col = first_existing_column(
        cols,
        [
            "rule_weight",
            "weight_rule",
            "w_rule",
            "clinical_weight",
            "weight_clinical",
        ],
    )

    required = {
        "target": target_col,
        "threshold": threshold_col,
        "l1_weight": l1_weight_col,
        "catboost_weight": cat_weight_col,
    }

    missing_fields = [
        name for name, col in required.items()
        if col is None
    ]

    if missing_fields:
        raise ValueError(
            "Не удалось распознать ensemble metadata.\n"
            f"Не найдены поля: {missing_fields}\n"
            f"Доступные колонки: {cols}"
        )

    settings = {}

    for target in targets:

        rows = meta.loc[
            meta[target_col].astype(str).eq(str(target))
        ].copy()

        if len(rows) == 0:
            raise ValueError(
                f"Для target={target} не найдено ни одной строки."
            )

        # В 04 настройки были подобраны отдельно
        # для каждого outer fold.
        # Для final deployable model фиксируем
        # медиану fold-specific settings.
        l1_weight = float(
            rows[l1_weight_col].median()
        )

        cat_weight = float(
            rows[cat_weight_col].median()
        )

        rule_weight = (
            float(rows[rule_weight_col].median())
            if rule_weight_col is not None
            else 0.0
        )

        threshold = float(
            rows[threshold_col].median()
        )

        ml_weight_sum = l1_weight + cat_weight

        if ml_weight_sum <= 0:
            raise ValueError(
                f"{target}: сумма L1 + CatBoost weights <= 0."
            )

        settings[target] = {
            "l1_weight": l1_weight,
            "catboost_weight": cat_weight,
            "rule_weight": rule_weight,
            "threshold": threshold,
            "n_folds": int(len(rows)),
        }

    return settings


ENSEMBLE_SETTINGS = extract_ensemble_settings(
    ensemble_meta,
    TARGETS,
)

settings_table = pd.DataFrame([
    {"target": target, **values}
    for target, values in ENSEMBLE_SETTINGS.items()
])

display(settings_table)


,target,l1_weight,catboost_weight,rule_weight,threshold,n_folds
0,iron_deficiency,0.4,0.6,0.0,0.41,5
1,B12_deficiency,0.5,0.5,0.0,0.26,5
2,folate_deficiency,0.5,0.5,0.0,0.45,5
3,B6_deficiency,0.3,0.7,0.0,0.45,5
4,copper_deficiency,0.6,0.4,0.0,0.18,5
5,inflammation_anemia,0.6,0.4,0.0,0.38,5


## 4. Формируем внешний X

Важно: мы передаём модели **ровно те же 37 колонок и в том же порядке**.

Отсутствующие в NHANES показатели уже представлены как `NaN`.


In [9]:
missing_features = [
    feature for feature in FEATURE_COLUMNS
    if feature not in external_df.columns
]

if missing_features:
    raise ValueError(
        f"В external matrix отсутствуют frozen features: {missing_features}"
    )

X_external = external_df[FEATURE_COLUMNS].copy()

print("X_external shape:", X_external.shape)
print(
    "Median non-missing features:",
    X_external.notna().sum(axis=1).median()
)


X_external shape: (11933, 37)
Median non-missing features: 16.0


## 5. Возрастной overlap с внутренней выборкой

NHANES содержит детей и взрослых, а внутренняя выборка имеет свой диапазон возраста.

Мы не будем скрывать этот domain shift.

Создаём флаг `age_in_internal_range` и дальше покажем результаты:

- для всей внешней выборки;
- отдельно внутри возрастного диапазона training data.


In [10]:
internal_age = pd.to_numeric(
    internal_df["age_years"],
    errors="coerce",
)

external_age = pd.to_numeric(
    external_df["age_years"],
    errors="coerce",
)

internal_age_min = float(internal_age.min())
internal_age_max = float(internal_age.max())

external_df["age_in_internal_range"] = (
    external_age.between(
        internal_age_min,
        internal_age_max,
        inclusive="both",
    )
)

print(
    f"Internal age range: "
    f"{internal_age_min:g}–{internal_age_max:g}"
)

print(
    "External patients inside internal age range:",
    int(external_df["age_in_internal_range"].sum()),
    "/",
    len(external_df),
)


Internal age range: 18–93
External patients inside internal age range: 8153 / 11933


## 6. Вспомогательные функции для frozen inference


In [11]:
EPS = 1e-6


def probability_to_logit(probabilities):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        EPS,
        1 - EPS,
    )
    return np.log(p / (1 - p)).reshape(-1, 1)


def apply_platt_calibrator(calibrator, probabilities):
    return calibrator.predict_proba(
        probability_to_logit(probabilities)
    )[:, 1]


def prepare_catboost_frame(
    frame: pd.DataFrame,
    feature_columns,
    categorical_features,
):
    out = frame[feature_columns].copy()

    for col in categorical_features:
        out[col] = out[col].astype("object")
        out[col] = out[col].where(
            out[col].notna(),
            "__MISSING__",
        )
        out[col] = out[col].astype(str)

    return out


## 7. Frozen inference: L1 + CatBoost + calibration + ensemble

Clinical-rule weight в `04` оказался нулевым, поэтому external probability
должна определяться frozen ML ensemble.

Если в metadata rule weight не равен нулю, notebook остановится:
для NHANES мы не хотим случайно смешать incomplete clinical-rule evidence
с ML probability.


In [12]:
external_predictions = pd.DataFrame({
    "external_patient_id": external_df["external_patient_id"],
    "age_in_internal_range": external_df["age_in_internal_range"],
})

X_external_cat = prepare_catboost_frame(
    X_external,
    FEATURE_COLUMNS,
    CATEGORICAL_FEATURES,
)

for target in TARGETS:
    print("\n" + "=" * 80)
    print("TARGET:", target)

    target_dir = MODEL_DIR / target

    required = [
        target_dir / "l1_pipeline.joblib",
        target_dir / "l1_platt_calibrator.joblib",
        target_dir / "catboost_model.cbm",
        target_dir / "catboost_platt_calibrator.joblib",
    ]

    missing_target_files = [
        str(p) for p in required if not p.exists()
    ]

    if missing_target_files:
        raise FileNotFoundError(
            f"Bundle incomplete for {target}:\n"
            + "\n".join(missing_target_files)
        )

    settings = ENSEMBLE_SETTINGS[target]

    if abs(settings["rule_weight"]) > 1e-12:
        raise ValueError(
            f"{target}: frozen rule_weight="
            f"{settings['rule_weight']}. "
            "Part 4 рассчитан на зафиксированный ML ensemble "
            "с rule_weight=0. Проверь metadata."
        )

    # L1
    l1_model = joblib.load(
        target_dir / "l1_pipeline.joblib"
    )
    l1_calibrator = joblib.load(
        target_dir / "l1_platt_calibrator.joblib"
    )

    l1_raw = l1_model.predict_proba(
        X_external
    )[:, 1]

    l1_cal = apply_platt_calibrator(
        l1_calibrator,
        l1_raw,
    )

    # CatBoost
    cat_model = CatBoostClassifier()
    cat_model.load_model(
        str(target_dir / "catboost_model.cbm")
    )

    cat_raw = cat_model.predict_proba(
        X_external_cat
    )[:, 1]

    cat_calibrator = joblib.load(
        target_dir / "catboost_platt_calibrator.joblib"
    )

    cat_cal = apply_platt_calibrator(
        cat_calibrator,
        cat_raw,
    )

    # Frozen weighted ML ensemble
    w_l1 = settings["l1_weight"]
    w_cat = settings["catboost_weight"]

    denominator = w_l1 + w_cat

    ensemble_prob = (
        w_l1 * l1_cal
        + w_cat * cat_cal
    ) / denominator

    threshold = settings["threshold"]
    ensemble_pred = (
        ensemble_prob >= threshold
    ).astype(int)

    external_predictions[f"{target}__l1_raw"] = l1_raw
    external_predictions[f"{target}__l1_cal"] = l1_cal
    external_predictions[f"{target}__cat_raw"] = cat_raw
    external_predictions[f"{target}__cat_cal"] = cat_cal
    external_predictions[f"{target}__prob"] = ensemble_prob
    external_predictions[f"{target}__pred"] = ensemble_pred

    print(
        f"threshold={threshold:.4f} | "
        f"positive predictions={ensemble_pred.sum()} "
        f"({ensemble_pred.mean():.3%})"
    )

print("\nFrozen external inference complete.")



TARGET: iron_deficiency
threshold=0.4100 | positive predictions=1340 (11.229%)

TARGET: B12_deficiency
threshold=0.2600 | positive predictions=204 (1.710%)

TARGET: folate_deficiency
threshold=0.4500 | positive predictions=49 (0.411%)

TARGET: B6_deficiency
threshold=0.4500 | positive predictions=0 (0.000%)

TARGET: copper_deficiency
threshold=0.1800 | positive predictions=0 (0.000%)

TARGET: inflammation_anemia
threshold=0.3800 | positive predictions=153 (1.282%)

Frozen external inference complete.


## 8. Feature coverage per patient

Поскольку максимум schema coverage сейчас 19/37,
обычный production threshold 0.70 здесь недостижим.

Поэтому NHANES — это именно **stress test under partial feature availability**.

Мы не будем выдавать эти прогнозы за полноценный clinical deployment result.


In [13]:
external_predictions["non_missing_features"] = (
    X_external.notna().sum(axis=1)
)

external_predictions["feature_coverage"] = (
    external_predictions["non_missing_features"]
    / len(FEATURE_COLUMNS)
)

coverage_bins = pd.cut(
    external_predictions["feature_coverage"],
    bins=[-0.001, 0.25, 0.40, 0.50, 1.0],
    labels=[
        "<=25%",
        "25–40%",
        "40–50%",
        ">50%",
    ],
)

external_predictions["coverage_bin"] = coverage_bins

coverage_distribution = (
    external_predictions["coverage_bin"]
    .value_counts(dropna=False)
    .rename_axis("coverage_bin")
    .reset_index(name="n")
)

coverage_distribution["pct"] = (
    coverage_distribution["n"]
    / len(external_predictions)
    * 100
)

display(coverage_distribution)


,coverage_bin,n,pct
0,40–50%,5370,45.001257
1,<=25%,4340,36.369731
2,>50%,1241,10.399732
3,25–40%,982,8.229280


## 9. External prediction prevalence

Сравниваем:

- frozen internal OOF predicted prevalence;
- external NHANES predicted prevalence;
- NHANES prevalence внутри training-age range.

Это **не accuracy metric** — это transfer sanity check.


In [14]:
internal_oof_path = (
    PROJECT_ROOT
    / "artifacts"
    / "ensemble"
    / "ensemble_oof_predictions.csv"
)

internal_oof = pd.read_csv(internal_oof_path)

prevalence_rows = []

for target in TARGETS:
    pred_candidates = [
        f"{target}__prediction",
        f"{target}__pred",
        f"{target}_prediction",
        f"{target}_pred",
    ]

    internal_pred_col = next(
        (
            col for col in pred_candidates
            if col in internal_oof.columns
        ),
        None,
    )

    external_pred_col = f"{target}__pred"

    row = {
        "target": target,
        "internal_oof_predicted_prevalence": (
            float(internal_oof[internal_pred_col].mean())
            if internal_pred_col is not None
            else np.nan
        ),
        "external_all_predicted_prevalence": float(
            external_predictions[external_pred_col].mean()
        ),
        "external_age_overlap_predicted_prevalence": float(
            external_predictions.loc[
                external_predictions["age_in_internal_range"],
                external_pred_col,
            ].mean()
        ),
    }

    prevalence_rows.append(row)

prediction_prevalence = pd.DataFrame(
    prevalence_rows
)

display(prediction_prevalence)


,target,internal_oof_predicted_prevalence,external_all_predicted_prevalence,external_age_overlap_predicted_prevalence
0,iron_deficiency,0.341667,0.112294,0.099718
1,B12_deficiency,0.228571,0.017095,0.023059
2,folate_deficiency,0.163095,0.004106,0.005887
3,B6_deficiency,0.034524,0.000000,0.000000
4,copper_deficiency,0.044048,0.000000,0.000000
5,inflammation_anemia,0.078571,0.012822,0.016681


## 10. Missingness shift: internal vs NHANES

Положительное `missingness_shift_pp` означает,
что во внешней выборке показатель отсутствует чаще.


In [15]:
missingness_rows = []

for feature in FEATURE_COLUMNS:
    internal_missing = float(
        internal_df[feature].isna().mean()
    )

    external_missing = float(
        X_external[feature].isna().mean()
    )

    missingness_rows.append({
        "feature": feature,
        "internal_missing_pct": internal_missing * 100,
        "external_missing_pct": external_missing * 100,
        "missingness_shift_pp": (
            external_missing - internal_missing
        ) * 100,
    })

missingness_shift = pd.DataFrame(
    missingness_rows
).sort_values(
    "missingness_shift_pp",
    ascending=False,
)

display(missingness_shift)


,feature,internal_missing_pct,external_missing_pct,missingness_shift_pp
31,eGFR,35.595238,100.000000,64.404762
16,UIBC,35.714286,100.000000,64.285714
15,TIBC,35.714286,100.000000,64.285714
17,TSAT,35.714286,100.000000,64.285714
14,transferrin,35.714286,100.000000,64.285714
20,vitamin_B12,38.928571,100.000000,61.071429
29,ESR,41.428571,100.000000,58.571429
12,ferritin,29.285714,83.658761,54.373047
32,TSH,49.404762,100.000000,50.595238
11,reticulocytes,52.261905,100.000000,47.738095


## 11. Numeric distribution shift

Для признаков, которые реально присутствуют в обеих выборках,
считаем двухвыборочный KS statistic.

- `KS ≈ 0` — распределения похожи;
- чем ближе к 1 — тем сильнее shift.

P-value здесь вторичен: при больших выборках даже небольшое отличие может быть статистически значимым.


In [16]:
distribution_shift_rows = []

for feature in FEATURE_COLUMNS:
    internal_values = pd.to_numeric(
        internal_df[feature],
        errors="coerce",
    ).dropna()

    external_values = pd.to_numeric(
        X_external[feature],
        errors="coerce",
    ).dropna()

    if len(internal_values) < 20 or len(external_values) < 20:
        continue

    ks = ks_2samp(
        internal_values,
        external_values,
    )

    iqr_internal = (
        internal_values.quantile(0.75)
        - internal_values.quantile(0.25)
    )

    if iqr_internal > 0:
        median_shift_iqr = (
            external_values.median()
            - internal_values.median()
        ) / iqr_internal
    else:
        median_shift_iqr = np.nan

    distribution_shift_rows.append({
        "feature": feature,
        "internal_n": len(internal_values),
        "external_n": len(external_values),
        "internal_median": float(internal_values.median()),
        "external_median": float(external_values.median()),
        "median_shift_in_internal_IQR": float(median_shift_iqr),
        "ks_statistic": float(ks.statistic),
        "ks_pvalue": float(ks.pvalue),
    })

distribution_shift = pd.DataFrame(
    distribution_shift_rows
).sort_values(
    "ks_statistic",
    ascending=False,
)

display(distribution_shift)


,feature,internal_n,external_n,internal_median,external_median,median_shift_in_internal_IQR,ks_statistic,ks_pvalue
17,LDH,325,4740,237.000,160.00,-0.675439,0.604778,2.328844e-106
6,MCHC,840,7593,322.600,338.00,0.916667,0.554311,2.727034e-217
12,sTfR,252,1949,1.915,3.16,0.618634,0.531895,5.432288e-59
1,hemoglobin,840,7593,111.700,137.00,0.681023,0.508269,1.843902e-180
13,folate,455,7348,6.000,15.00,1.132075,0.479892,5.927865e-91
7,RDW,825,7593,15.500,13.50,-0.512821,0.466670,2.817422e-148
2,RBC,840,7593,4.000,4.66,0.510638,0.445355,1.233183e-136
3,hematocrit,840,7593,35.100,40.60,0.543210,0.433941,1.946476e-129
0,age_years,840,11933,60.000,37.00,-0.766667,0.353596,2.677671e-88
11,serum_iron,540,6364,11.650,15.00,0.310185,0.271451,7.414403e-33


## 12. Model disagreement on NHANES

Если L1 и CatBoost сильно расходятся на внешних данных,
это дополнительный сигнал domain shift / low confidence.

Считаем абсолютную разницу между откалиброванными вероятностями.


In [17]:
disagreement_rows = []

for target in TARGETS:
    diff = (
        external_predictions[f"{target}__l1_cal"]
        - external_predictions[f"{target}__cat_cal"]
    ).abs()

    disagreement_rows.append({
        "target": target,
        "median_abs_l1_cat_diff": float(diff.median()),
        "q75_abs_l1_cat_diff": float(diff.quantile(0.75)),
        "q90_abs_l1_cat_diff": float(diff.quantile(0.90)),
        "max_abs_l1_cat_diff": float(diff.max()),
    })

model_disagreement = pd.DataFrame(
    disagreement_rows
)

display(model_disagreement)


,target,median_abs_l1_cat_diff,q75_abs_l1_cat_diff,q90_abs_l1_cat_diff,max_abs_l1_cat_diff
0,iron_deficiency,0.014047,0.033646,0.081505,0.993646
1,B12_deficiency,0.008721,0.015949,0.076041,0.997733
2,folate_deficiency,0.005110,0.023915,0.024344,0.535315
3,B6_deficiency,0.007604,0.049453,0.054978,0.342628
4,copper_deficiency,0.001888,0.014279,0.017747,0.017747
5,inflammation_anemia,0.000520,0.001312,0.002672,0.999843


## 13. Confidence-like transfer summary

Это **не clinical confidence** из Expert System `05`.

Здесь мы только смотрим distance от frozen decision threshold:
чем ближе вероятность к threshold, тем менее устойчиво binary decision.


In [18]:
threshold_margin_rows = []

for target in TARGETS:
    prob = external_predictions[f"{target}__prob"]
    threshold = ENSEMBLE_SETTINGS[target]["threshold"]

    margin = (prob - threshold).abs()

    threshold_margin_rows.append({
        "target": target,
        "threshold": threshold,
        "median_abs_margin": float(margin.median()),
        "pct_within_0_05_of_threshold": float(
            (margin <= 0.05).mean() * 100
        ),
        "pct_within_0_10_of_threshold": float(
            (margin <= 0.10).mean() * 100
        ),
    })

threshold_margin_summary = pd.DataFrame(
    threshold_margin_rows
)

display(threshold_margin_summary)


,target,threshold,median_abs_margin,pct_within_0_05_of_threshold,pct_within_0_10_of_threshold
0,iron_deficiency,0.41,0.388676,0.796112,1.433001
1,B12_deficiency,0.26,0.254818,1.156457,2.321294
2,folate_deficiency,0.45,0.446961,0.100561,0.259784
3,B6_deficiency,0.45,0.437088,0.000000,0.000000
4,copper_deficiency,0.18,0.178453,0.000000,0.000000
5,inflammation_anemia,0.38,0.379766,0.460907,0.879913


## 14. Сохраняем Part 4 artifacts


In [19]:
external_predictions_path = (
    EXTERNAL_ARTIFACTS_DIR
    / "nhanes_frozen_predictions.csv"
)

prediction_prevalence_path = (
    EXTERNAL_ARTIFACTS_DIR
    / "nhanes_prediction_prevalence.csv"
)

missingness_shift_path = (
    EXTERNAL_ARTIFACTS_DIR
    / "nhanes_missingness_shift.csv"
)

distribution_shift_path = (
    EXTERNAL_ARTIFACTS_DIR
    / "nhanes_distribution_shift.csv"
)

model_disagreement_path = (
    EXTERNAL_ARTIFACTS_DIR
    / "nhanes_model_disagreement.csv"
)

threshold_margin_path = (
    EXTERNAL_ARTIFACTS_DIR
    / "nhanes_threshold_margin_summary.csv"
)

external_predictions.to_csv(
    external_predictions_path,
    index=False,
)

prediction_prevalence.to_csv(
    prediction_prevalence_path,
    index=False,
)

missingness_shift.to_csv(
    missingness_shift_path,
    index=False,
)

distribution_shift.to_csv(
    distribution_shift_path,
    index=False,
)

model_disagreement.to_csv(
    model_disagreement_path,
    index=False,
)

threshold_margin_summary.to_csv(
    threshold_margin_path,
    index=False,
)

print("Saved:", external_predictions_path)
print("Saved:", prediction_prevalence_path)
print("Saved:", missingness_shift_path)
print("Saved:", distribution_shift_path)
print("Saved:", model_disagreement_path)
print("Saved:", threshold_margin_path)


Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_frozen_predictions.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_prediction_prevalence.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_missingness_shift.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_distribution_shift.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_model_disagreement.csv
Saved: /home/sinopah/meditron/artifacts/external_validation/nhanes_threshold_margin_summary.csv


# Что прислать после запуска Part 4

Пришли:

1. таблицу `settings_table`;
2. `Internal age range` и число NHANES patients inside internal age range;
3. строки с positive predictions для всех 6 targets;
4. таблицу `coverage_distribution`;
5. `prediction_prevalence`;
6. первые 15 строк `missingness_shift`;
7. `distribution_shift`;
8. `model_disagreement`;
9. `threshold_margin_summary`.

Если parser ensemble metadata остановится в секции 3 —
просто пришли `columns:` и саму таблицу `ensemble_meta_settings`.

После проверки этих результатов сделаем **Part 5 — финальный external-validation report**
и отдельно сформулируем, что именно можно честно заявить жюри как external validation,
а что является stress-test / domain-shift analysis.
